In [1]:
import pandas as pd

df = pd.read_csv(
    r"C:\Users\jai ganesh\Documents\sla_analysis\data\delivery_order.csv"
)

df.head()

,order_id,store_id,city,order_placed_time,order_delivered_time,delivery_minutes,distance_km,promised_sla_minutes,breached,day_of_week,hour_of_day
0,ORD-000001,STR-001,Jalandhar,2026-06-01 08:26:58,2026-06-01 08:37:11.200,10.22,0.81,10,True,Monday,8
1,ORD-000002,STR-001,Jalandhar,2026-06-01 15:51:56,2026-06-01 16:00:47.000,8.85,0.26,10,False,Monday,15
2,ORD-000003,STR-001,Jalandhar,2026-06-01 11:38:17,2026-06-01 11:49:51.200,11.57,3.77,10,True,Monday,11
3,ORD-000004,STR-001,Jalandhar,2026-06-01 10:19:27,2026-06-01 10:30:36.000,11.15,0.90,10,True,Monday,10
4,ORD-000005,STR-001,Jalandhar,2026-06-01 17:46:34,2026-06-01 17:56:09.400,9.59,3.08,10,False,Monday,17


## “How much of the company's total SLA breach problem comes from these three stores?”

In [2]:
chronic = ['STR-001', 'STR-002', 'STR-003']

total_breaches = df['breached'].sum()

chronic_breaches = df[
    df['store_id'].isin(chronic)
]['breached'].sum()

share = chronic_breaches / total_breaches

print("Total breaches:", total_breaches)
print("Chronic store breaches:", chronic_breaches)
print("Share of all breaches:", f"{share * 100:.2f}%")

Total breaches: 20686
Chronic store breaches: 10992
Share of all breaches: 53.14%


## Simulate the impact of fixing the 3 chronic stores

In [3]:
orders = df.groupby('store_id').size()

rates = df.groupby('store_id')['breached'].mean()

avg_rate = rates.drop(chronic).mean()

new_rates = rates.copy()

new_rates[chronic] = avg_rate

new_overall = (new_rates * orders).sum() / orders.sum()

print("Before:", f"{df['breached'].mean() * 100:.2f}%")
print("After: ", f"{new_overall * 100:.2f}%")

Before: 41.42%
After:  26.01%


## Impact Simulation: If STR-001, STR-002, and STR-003 improved to the average breach rate of the remaining stores, the overall breach rate would decrease from [your Before value]% to [your After value]%, an improvement of [your improvement] percentage points.

In [4]:
improvement = df['breached'].mean() - new_overall

print("Before:", f"{df['breached'].mean() * 100:.2f}%")
print("After:", f"{new_overall * 100:.2f}%")
print("Improvement:", f"{improvement * 100:.2f} percentage points")

Before: 41.42%
After: 26.01%
Improvement: 15.41 percentage points


# Delivery SLA Breach Recommendations

Based on the analysis of delivery SLA breaches, the following actions are recommended:

## 1. Staffing and Process Audit at Chronic-Breach Stores

STR-001, STR-002, and STR-003 show consistently high breach rates, including during quieter hours. This suggests that the issue may not be caused only by peak-hour demand.

Recommended actions:
- Review staffing levels and shift schedules.
- Check order preparation and handoff times.
- Review rider availability and allocation.
- Identify recurring operational bottlenecks at these stores.

## 2. Rebalance Rider Allocation During Peak Hours

The analysis shows higher breach risk during busy periods, particularly around 8–10 PM.

Recommended action:
- Monitor rider demand at STR-001, STR-002, and STR-003 during 8–10 PM.
- Temporarily allocate additional riders to stores with high demand and breach rates.
- Compare breach rates before and after the allocation change.

## 3. Create a Rolling 7-Day Breach Alert

Create an automated alert when a store's rolling 7-day breach rate crosses 60%.

Example rule:

`Rolling 7-day breach rate > 60% → Trigger alert`

This allows the operations team to investigate a store before the problem becomes persistent.

## 4. Use the Risk Model for High-Risk Orders

The Random Forest model can be used to identify orders with a high predicted probability of SLA breach.

For high-risk orders, the business could consider operational interventions such as:
- Prioritizing the order for preparation or rider assignment.
- Monitoring the order more closely.
- Where business rules permit, providing a more realistic promised delivery time when the predicted breach probability is high.

The model should be monitored regularly to ensure that its predictions remain accurate and useful.
## Limitations

- **Simulated data:** This project uses simulated delivery data, so real-world factors such as traffic, weather, stock-outs, and unexpected operational issues are not captured.
- **Correlation vs. causation:** The analysis identifies patterns and relationships in the data, but it does not prove that a specific factor directly causes SLA breaches.
- **Operational changes need testing:** Before making major staffing or rider-allocation changes, a real-world A/B test or controlled pilot should be conducted to measure the actual impact.
- **Model limitations:** The risk model is based on the available features. Its predictions should be monitored and validated with real operational data before being used for production decisions.